# 05 — Full Fusion Pipeline
Run the complete LiDAR-Camera fusion pipeline across multiple KITTI frames.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics opencv-python matplotlib -q

In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

BASE_PATH  = '/content/drive/MyDrive/Project/Kitti/Training'
IMAGE_PATH = os.path.join(BASE_PATH, 'image_2/training/image_2')
LIDAR_PATH = os.path.join(BASE_PATH, 'velodyne/training/velodyne')
CALIB_PATH = os.path.join(BASE_PATH, 'calib/data_object_calib/training/calib')
RESULTS    = '/content/drive/MyDrive/Project/results'
os.makedirs(RESULTS, exist_ok=True)

def parse_calib(calib_path):
    calib = {}
    with open(calib_path, 'r') as f:
        for line in f.readlines():
            if ':' in line:
                key, value = line.split(':', 1)
                calib[key.strip()] = np.array(
                    [float(x) for x in value.strip().split()])
    return (calib['P2'].reshape(3,4),
            calib['R0_rect'].reshape(3,3),
            calib['Tr_velo_to_cam'].reshape(3,4))

def project_lidar_to_image(points_3d, P2, R0, Tr):
    N       = points_3d.shape[0]
    pts_hom = np.hstack([points_3d, np.ones((N,1))])
    Tr_full = np.vstack([Tr,[0,0,0,1]])
    R0_full = np.eye(4); R0_full[:3,:3] = R0
    pts_cam = R0_full @ Tr_full @ pts_hom.T
    depth   = pts_cam[2,:]; valid = depth > 0
    pts_cam = pts_cam[:,valid]; depth = depth[valid]
    pts_img = P2 @ pts_cam; pts_img = pts_img/pts_img[2,:]
    return pts_img[:2,:].T, depth, valid

def get_lidar_depth_for_box(box_2d, pixels, depth, pts_xyz, valid_mask):
    x1,y1,x2,y2 = box_2d
    px,py = pixels[:,0], pixels[:,1]
    in_box = (px>=x1)&(px<=x2)&(py>=y1)&(py<=y2)
    d = depth[in_box]
    if len(d) == 0: return None, 0
    med   = np.median(d)
    clean = d[np.abs(d-med)<2.0]
    if len(clean) == 0: return None, 0
    return round(float(clean.mean()),2), len(clean)

def process_frame(sample_id, model, conf=0.3):
    img_path   = f'{IMAGE_PATH}/{sample_id}.png'
    lidar_path = f'{LIDAR_PATH}/{sample_id}.bin'
    calib_path = f'{CALIB_PATH}/{sample_id}.txt'
    if not all(os.path.exists(p) for p in [img_path,lidar_path,calib_path]):
        return None
    img     = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h,w     = img.shape[:2]
    pts     = np.fromfile(lidar_path, dtype=np.float32).reshape(-1,4)
    pts_xyz = pts[:,:3]
    P2,R0,Tr = parse_calib(calib_path)
    pixels,depth,valid = project_lidar_to_image(pts_xyz,P2,R0,Tr)
    px,py = pixels[:,0],pixels[:,1]
    in_bounds = (px>=0)&(px<w)&(py>=0)&(py<h)
    results   = model(img_path,conf=conf,classes=[0,1,2,3,5,7],verbose=False)
    boxes     = results[0].boxes.xyxy.cpu().numpy()
    scores    = results[0].boxes.conf.cpu().numpy()
    class_ids = results[0].boxes.cls.cpu().numpy()
    class_names = {0:'person',1:'bicycle',2:'car',3:'motorcycle',5:'bus',7:'truck'}
    colors      = {0:'red',1:'orange',2:'lime',3:'cyan',5:'magenta',7:'yellow'}
    fig,ax = plt.subplots(1,figsize=(16,6))
    ax.imshow(img_rgb)
    dn = (depth-depth.min())/(depth.max()-depth.min())
    ax.scatter(px[in_bounds],py[in_bounds],c=dn[in_bounds],cmap='jet',s=1.5,alpha=0.4)
    detections = []
    for box,score,cls in zip(boxes,scores,class_ids):
        x1,y1,x2,y2 = box
        cls_int = int(cls)
        name    = class_names.get(cls_int,'unknown')
        color   = colors.get(cls_int,'white')
        dist,n  = get_lidar_depth_for_box(box,pixels,depth,pts_xyz,valid)
        label   = f'{name} {score:.2f} | {dist}m' if dist else f'{name} {score:.2f}'
        rect    = patches.Rectangle((x1,y1),x2-x1,y2-y1,
                                     linewidth=2,edgecolor=color,facecolor='none')
        ax.add_patch(rect)
        ax.text(x1,y1-5,label,color=color,fontsize=9,fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',facecolor='black',alpha=0.6))
        detections.append({'class':name,'confidence':float(score),
                           'box':box.tolist(),'distance_m':dist,'lidar_points':n})
    ax.set_title(f'LiDAR-Camera Fusion — Sample {sample_id}',fontsize=13)
    ax.axis('off'); plt.tight_layout()
    plt.savefig(f'{RESULTS}/{sample_id}_fusion.png',dpi=150,bbox_inches='tight')
    plt.show(); plt.close()
    return detections

model    = YOLO('yolov8n.pt')
all_ids  = sorted([f.replace('.png','') for f in os.listdir(IMAGE_PATH) if f.endswith('.png')])
sample_ids = all_ids[::750][:10]

for sid in sample_ids:
    print(f'Processing {sid}...')
    dets = process_frame(sid, model)
    if dets:
        for d in dets:
            print(f"  {d['class']:10s} conf={d['confidence']:.2f} dist={d['distance_m']}m")
    print()